# TusoAI PGBoost repo runner

This notebook configures TusoAI to optimize a method that lives in the PGBoost example codebase.


In [ ]:
import os
from tusoai import Tusoai

# Set OPENAI_API_KEY in your environment before running this notebook.
# Set SEMANTIC_SCHOLAR_API_KEY optionally for higher-throughput literature search.
semantic_scholar_api_key = os.environ.get("SEMANTIC_SCHOLAR_API_KEY")

ai = Tusoai.from_api_key(
    api_key=os.environ["OPENAI_API_KEY"],
    provider="openai",
    temperature=1.0,
    max_tokens=15000,
    model_settings={
        # PDF parsing / summarization model
        "pdf": {
            "model": "gpt-5.4-nano",
            "thinking": True,
            "thinking_tokens": 5000,
            "reasoning_mode": "medium",
        },
        # Knowledge-tree/instruction construction model
        "construction": {
            "model": "gpt-5.4",
            "thinking": True,
            "thinking_tokens": 5000,
            "reasoning_mode": "medium",
        },
        # Optimization/mutation model
        "optimization": {
            "model": "gpt-5.4-nano",
            "thinking": True,
            "thinking_tokens": 5000,
            "reasoning_mode": "medium",
        },
    },
)


In [ ]:
task_description = "linking regulatory variants to target genes"
data_available   = "predefined genomic features of SNP-gene pairs from various tool outputs"
cache_dir = "tusoai_pgboost_all"

# --- Build the method subtask ---
function_name = "pgBoost_method"

method_task, method_cost = ai.create_method_subtask(
    function_name=function_name,
    task_description=task_description,
    data_available=data_available,
    num_cat=10,
    instruction_count=20,
    num_init=10,
    paper_searches=10,
    info_per_paper=20,
    clear=False,
    cache_dir=cache_dir,
    semantic_scholar_api_key=semantic_scholar_api_key,
    hints=[
        "Feel free to make drastic changes if needed.",
        "Keep your implementation principled and efficient.",
        "Keep the function header and output shape unchanged.",
    ],
    use_initial=True,
    #source_path="pgboost_method.py",
    #repo_root="pgboost_github2",
)


In [ ]:
method_cost

In [ ]:
# --- Build the data subtask ---
read_cmd = """
data = pd.read_csv("pgboost/gencode_data/gencode.v49.annotation.gtf.gz", sep="\t", comment="#", header=None, compression="gzip").set_axis(["chr","source","feature","start","end","score","strand","frame","attribute"], axis=1)
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 'gencode_features'
file_description = 'Gencode gene annotations'
gencode_data_hints = ["In features_df, each row is a SNP-gene pair and their associated features.",
 "features_df has columns 'chr' for chromosome id, 'gene' for gene name, 'snp' for snp name, 'tss' for gene TSS, 'tes' for gene TES, and 'snp_position'.Use the above information to construct new SNP-gene linking features for each row using the new genomic data.",
 "Add any new features as columns in features_df. Get gene information using features_df[\"gene\"], obtain gene information from the new file, and then build new features using the snp_position relative to that new gene information.",
 "IMPORTANT: Add/edit just one new feature.",
 "IMPORTANT: Do not do anything that would add more rows to features_df.",
 "IMPORTANT: New features must be numerical.",
 "Keep the function header, input, and output the same."]

data_task, data_cost = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)


In [ ]:
data_cost

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/gencode_data/GRCh38.primary_assembly.genome.fa.gz", sep="\t", comment=">", header=None, compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 'gencode_grch38_primary_assembly_genome_features'
file_description = 'Gencode GRCh38 primary assembly genome sequence'
data_task2, data_cost2 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)
data_cost2

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/gencode_data/gencode.v49.polyAs.gtf.gz", sep="\t", comment="#", header=None, compression="gzip").set_axis(["chr","source","feature","start","end","score","strand","frame","attribute"], axis=1)
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 'gencode_v49_polyas_features'
file_description = 'Gencode polyA annotations'
data_task3, data_cost3 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)
data_cost3

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/gencode_data/gencode.v49.pc_transcripts.fa.gz", sep="\t", comment=">", header=None, compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 'gencode_v49_pc_transcripts_features'
file_description = 'Gencode protein-coding transcript sequences'
data_task4, data_cost4 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)
data_cost4

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/gencode_data/gencode.v49.long_noncoding_RNAs.gtf.gz", sep="\t", comment="#", header=None, compression="gzip").set_axis(["chr","source","feature","start","end","score","strand","frame","attribute"], axis=1)
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 'gencode_v49_long_noncoding_rnas_features'
file_description = 'Gencode long noncoding RNA annotations'
data_task5, data_cost5 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)


In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/gencode_data/gencode.v49.lncRNA_transcripts.fa.gz", sep="\t", comment=">", header=None, compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 'gencode_v49_lncrna_transcripts_features'
file_description = 'Gencode lncRNA transcript sequences'
data_task6, data_cost6 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/gencode_data/gencode.v49.annotation.simple.tsv.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 'gencode_v49_annotation_simple_features'
file_description = 'Gencode simplified gene annotations'
data_task7, data_cost7 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/5000_blocks_30_hic.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_5000_blocks_30_hic_features'
file_description = 're2g 5kb Hi-C blocks'
data_task8, data_cost8 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/10000_blocks_30_hic.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_10000_blocks_30_hic_features'
file_description = 're2g 10kb Hi-C blocks'
data_task9, data_cost9 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/call-hiccups-shard-1-merged_loops_30.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_call_hiccups_shard_1_merged_loops_30_features'
file_description = 're2g merged HiCCUPS loop calls'
data_task10, data_cost10 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/distal_regulation_dnase_dnase_eg_correlations.predictions.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_distal_regulation_dnase_dnase_eg_correlations_predictions_features'
file_description = 're2g distal regulation DNase-DNase element-gene correlation predictions'
data_task11, data_cost11 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/dnase-h3k27ac-element-gene-link-predictions_hic.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_dnase_h3k27ac_element_gene_link_predictions_hic_features'
file_description = 're2g DNase-H3K27ac element-gene link predictions from Hi-C'
data_task12, data_cost12 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/dnase-h3k27ac-element-gene-link-predictions_hic_K562.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_dnase_h3k27ac_element_gene_link_predictions_hic_k562_features'
file_description = 're2g DNase-H3K27ac element-gene link predictions from Hi-C in K562'
data_task13, data_cost13 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)


In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/epimap_score_GM12878.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_epimap_score_gm12878_features'
file_description = 're2g EpiMap scores in GM12878'
data_task14, data_cost14 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/epimap_score_k562.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_epimap_score_k562_features'
file_description = 're2g EpiMap scores in K562'
data_task15, data_cost15 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/GraphRegLR_Features_Predictions_Thresholded.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_graphreglr_features_predictions_thresholded_features'
file_description = 're2g GraphRegLR thresholded feature predictions'
data_task16, data_cost16 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/GraphRegLR_Features_Predictions_Thresholded_K562.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_graphreglr_features_predictions_thresholded_k562_features'
file_description = 're2g GraphRegLR thresholded feature predictions in K562'
data_task17, data_cost17 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/GraphRegLR_Predictions.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_graphreglr_predictions_features'
file_description = 're2g GraphRegLR predictions'
data_task18, data_cost18 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/GraphRegLR_Predictions_K562.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_graphreglr_predictions_k562_features'
file_description = 're2g GraphRegLR predictions in K562'
data_task19, data_cost19 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/LHG0052H.e500.clusters.cis.BE3.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_lhg0052h_e500_clusters_cis_be3_features'
file_description = 're2g LHG0052H e500 cis cluster BE3 data'
data_task20, data_cost20 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/LHK0001N.e500.clusters.cis.BE3.gz", sep="\t", compression="gzip")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_lhk0001n_e500_clusters_cis_be3_features'
file_description = 're2g LHK0001N e500 cis cluster BE3 data'
data_task21, data_cost21 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/P2_promoter_class.txt", sep="\t")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_p2_promoter_class_features'
file_description = 're2g P2 promoter class annotations'
data_task22, data_cost22 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/per_gene_weighted_correlation_ABC_enhancers_across_cell_types.txt", sep="\t")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_per_gene_weighted_correlation_abc_enhancers_across_cell_types_features'
file_description = 're2g per-gene weighted correlation of ABC enhancers across cell types'
data_task23, data_cost23 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/phastCon30way_GM12878_ABC_windows.txt", sep="\t")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_phastcon30way_gm12878_abc_windows_features'
file_description = 're2g phastCons 30-way conservation scores for GM12878 ABC windows'
data_task24, data_cost24 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/phastCon30way_K562_ABC_windows.txt", sep="\t")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_phastcon30way_k562_abc_windows_features'
file_description = 're2g phastCons 30-way conservation scores for K562 ABC windows'
data_task25, data_cost25 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/phyloP_GM12878_ABC_windows.txt", sep="\t")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_phylop_gm12878_abc_windows_features'
file_description = 're2g phyloP conservation scores for GM12878 ABC windows'
data_task26, data_cost26 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/phyloP_K562_ABC_windows.txt", sep="\t")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_phylop_k562_abc_windows_features'
file_description = 're2g phyloP conservation scores for K562 ABC windows'
data_task27, data_cost27 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = "pgboost/re2g_data/pooled_x_CHIP.crop_28-2bp.nodup.pval.signal.bigWig"
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_pooled_x_chip_crop_28_2bp_nodup_pval_signal_features'
file_description = 're2g pooled ChIP p-value signal bigWig track'
data_task28, data_cost28 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = pd.read_csv("pgboost/re2g_data/Ubiquitous_expression.txt", sep="\t")
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_ubiquitous_expression_features'
file_description = 're2g ubiquitous expression annotations'
data_task29, data_cost29 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = "pgboost/re2g_data/CHIP.crop_28-2bp.srt.bam"
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_chip_crop_28_2bp_srt_bam_features'
file_description = 're2g ChIP BAM alignments cropped at 28-2bp'
data_task30, data_cost30 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)



In [ ]:
read_cmd = """
data = "pgboost/re2g_data/CHIP.crop_36-2bp.srt.bam"
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_chip_crop_36_2bp_srt_bam_features'
file_description = 're2g ChIP BAM alignments cropped at 36-2bp'
data_task31, data_cost31 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
read_cmd = """
data = "pgboost/re2g_data/CHIP_pool.pvalue_signal.bigWig"
"""

data_usage = 'create SNP-gene features'
data_instruction_count = 30
function_name = 're2g_chip_pool_pvalue_signal_features'
file_description = 're2g pooled ChIP p-value signal bigWig track'
data_task32, data_cost32 = ai.create_data_subtask(
    function_name=function_name,
    task_description=task_description,
    file_description=file_description,
    data_usage=data_usage,
    read_cmd=read_cmd,
    cache_dir=cache_dir,
    data_instruction_count=data_instruction_count,
    clear=False,
    hints=gencode_data_hints,
    #source_path="pgboost_runner.py",
    #repo_root="pgboost_github2",
)

In [ ]:
data_tasks = [data_task,
              data_task2,
              data_task3,
              data_task4,
              data_task5,
              data_task6,
              data_task7,
              data_task8,
              data_task9,
              data_task10,
              data_task11,
              data_task12,
              data_task13,
              data_task14,
              data_task15,
              data_task16,
              data_task17,
              data_task18,
              data_task19,
              data_task20,
              data_task21,
              data_task22,
              data_task23,
              data_task24,
              data_task25,
              data_task26,
              data_task27,
              data_task28,
              data_task29,
              data_task30,
              data_task31,
              data_task32
             ]

In [ ]:


              
# --- Run discovery ---
best_model, history = ai.optimize(
    method_tasks=[method_task],
    data_tasks=[],#data_tasks,
    reference_filename="pgboost/pgboost_410_method.py",
    timeout=1200,
    bug_retries=3,
    n_feedback_buffer=10,
    skip_timeout=True,
    prompt_samples=3,
    drop_island_iter=60,
    prompt_decay=2.0,
    prompt_importance=100.0,
    max_islands=1,
    #load_history="tusoai_pgboost_all/history/history_1775488187_i4kbu9/history.json",
    output_dir=cache_dir,
    TIME_LIMIT=72 * 60,
    task_description=task_description,
    debug=False,
    min_improvement=0.0025,
    n_jobs=4,
    COST_LIMIT=50,
)


In [ ]:
data_task